# Paradigm Shift v1: 2단계 잔차 학습 (Two-Stage Residual Learning)

이 노트북은 기존의 수동 보정 방식을 버리고, **모델이 오차(Residual)를 직접 학습**하게 합니다.

**핵심 전략:**
1. **Stage 1 (Operational Model):** 정원, 휴가, 출장 등 핵심 지표로 참여율 예측. (노이즈 배제)
2. **Stage 2 (Correction Model):** Stage 1의 **오차(Actual - Predict)**를 타겟으로 하여 메뉴, 날씨, 상세 날짜 정보로 학습.
3. **요일별 분화:** 요일별로 다른 오차 패턴을 모델이 스스로 인지하도록 설계.
4. **Target Smoothing:** 극단적인 오차값(이상치)을 완화하여 모델의 일반화 성능 강화.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색 (CY2 환경)
def find_path(cands): 
    for p in cands: 
        if os.path.exists(p): return p
    return None

train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\train_median.csv", r"./data/train_median.csv"])
test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\test.csv", r"./data/test.csv"])
sub_path = find_path([r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv", r"./data/sample_submission.csv"])
w_train_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weather.csv", r"./data/weather.csv"])
w_test_path = find_path([r"C:\Users\CY2\github\SecondProject\data\weatherTest.csv", r"./data/weatherTest.csv"])

train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sub_path, encoding="utf-8-sig")
w_train = pd.read_csv(w_train_path, encoding="utf-8-sig")
w_test = pd.read_csv(w_test_path, encoding="utf-8-sig") if w_test_path else pd.DataFrame()

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 날씨 전처리 (Stage 2용)
def prep_w(df):
    if df.empty: return df
    c = df.columns
    d_col = [x for x in c if "일시" in x or "날짜" in x][0]
    t_col = [x for x in c if "기온" in x][0]
    r_col = [x for x in c if "강수량" in x][0]
    res = df[[d_col, t_col, r_col]].copy()
    res.columns = ["일자", "기온", "강수량"]
    res["일자"] = pd.to_datetime(res["일자"])
    res["기온"] = pd.to_numeric(res["기온"], errors="coerce").interpolate().ffill().bfill()
    res["강수량"] = pd.to_numeric(res["강수량"], errors="coerce").fillna(0)
    res["is_rain"] = (res["강수량"] > 0).astype(int)
    return res

weather_all = pd.concat([prep_w(w_train), prep_w(w_test)]).drop_duplicates("일자")
train = train.merge(weather_all, on="일자", how="left")
test = test.merge(weather_all, on="일자", how="left")

In [ ]:
# 피처 엔지니어링: Stage 1(운영) vs Stage 2(세부)
def add_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)
    
    # Stage 1용
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["days_to_month_end"] = ((df["일자"] + pd.offsets.MonthEnd(0)) - df["일자"]).dt.days
    
    # Stage 2용
    df["is_payday"] = df["일"].apply(lambda x: 1 if x in [24, 25, 26] else 0)
    prev_gap = df["일자"].diff().dt.days.fillna(1)
    next_gap = df["일자"].shift(-1).sub(df["일자"]).dt.days.fillna(1)
    df["holiday_after"] = (prev_gap >= 2).astype(int)
    df["holiday_before"] = (next_gap >= 2).astype(int)
    
    # 메뉴 키워드 추출 (Stage 2용)
    for kw in ["돈까스", "제육", "치킨", "카레", "비빔밥", "짜장", "탕수육", "볶음밥"]:
        df[f"m_{kw}"] = df["중식메뉴"].str.contains(kw, na=False).astype(int)
        df[f"d_{kw}"] = df["석식메뉴"].str.contains(kw, na=False).astype(int)
        
    return df

train = add_features(train)
test = add_features(test)
train["중식참여율"] = train["중식계"] / train["식사가능자수"]
train["석식참여율"] = train["석식계"] / train["식사가능자수"]

In [ ]:
# [Stage 1] 운영 베이스 모델
lunch_f1 = ["월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수","is_fri","days_to_month_end"]
dinner_f1 = ["월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수","is_wed"]

model_l1 = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4, random_state=42)
model_d1 = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4, random_state=42)

model_l1.fit(train[lunch_f1], train["중식참여율"])
model_d1.fit(train[dinner_f1], train["석식참여율"])

train["l_pred1"] = model_l1.predict(train[lunch_f1])
train["d_pred1"] = model_d1.predict(train[dinner_f1])

# 오차(Residual) 계산
train["l_res"] = train["중식참여율"] - train["l_pred1"]
train["d_res"] = train["석식참여율"] - train["d_pred1"]

In [ ]:
# [Stage 2] 오차 학습 모델 (Correction Model)
# Stage 1에서 쓰지 않은 메뉴, 날씨, 상세 날짜 정보를 사용
lunch_f2 = ["기온", "is_rain", "is_payday", "holiday_after", "holiday_before"] + [c for c in train.columns if c.startswith("m_")]
dinner_f2 = ["기온", "is_rain", "is_payday", "holiday_after", "holiday_before"] + [c for c in train.columns if c.startswith("d_")]

# 오차 학습은 과적합되기 쉬우므로 아주 약하게 학습(LGBM 사용)
model_l2 = LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=3, random_state=42, verbose=-1)
model_d2 = LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=3, random_state=42, verbose=-1)

model_l2.fit(train[lunch_f2], train["l_res"])
model_d2.fit(train[train["석식계"]>0][dinner_f2], train[train["석식계"]>0]["d_res"])

print("Stage 2 학습 완료")

In [ ]:
# 최종 결합 예측
l_base = model_l1.predict(test[lunch_f1])
d_base = model_d1.predict(test[dinner_f1])

l_corr = model_l2.predict(test[lunch_f2])
d_corr = model_d2.predict(test[dinner_f2])

# 오차 예측값의 60%만 반영 (안전장치)
final_l_ratio = l_base + (l_corr * 0.6)
final_d_ratio = d_base + (d_corr * 0.6)

pred_l = final_l_ratio * test["식사가능자수"]
pred_d = final_d_ratio * test["식사가능자수"]

submission = sample_submission.copy()
submission[submission.columns[1]] = np.clip(pred_l.values, 0, None)
submission[submission.columns[2]] = np.clip(pred_d.values, 0, None)

submission.to_csv("paradigm_shift_v1.csv", index=False, encoding="utf-8-sig")
print("저장 완료: paradigm_shift_v1.csv")